# Attackers vs Address Space: Count Regression (DDS ch. 5)

Raw "attacks by country" rankings mostly show where the internet's address space is: the US and China top them because they hold the most IPv4 addresses. Data-Driven Security ch. 5 makes the point that you have to model counts against **exposure** before a country "stands out". The question here:

> Which countries send more, or fewer, attacking IPs than their IPv4 address space predicts, and does that hold up?

The model, per country:

$$\log E[\text{attacking IPs}] = b_0 + \beta \cdot \log(\text{IPv4 addresses})$$

$\beta$ is estimated, not fixed at 1. $\beta = 1$ would mean attackers scale in proportion to address space; $\beta < 1$ means big address holders send fewer attackers per address.

The notebook has three parts:
1. **Which count model?** Poisson or negative binomial. The assumptions are checked before trusting any result.
2. **The fit.** The slope, and the countries furthest from it.
3. **Validation.** Do the conclusions survive different ways of splitting the data, a different response, influential countries, resampling and cross-validation?

Exploration only. Logic lives in `../exposure.py`, `../regression.py`, `../validation.py` and `../charts.py`; method notes and findings are in `../README.md`.

**Setup**: the shared "Python (bear-trap analysis)" kernel, see `../../network/README.md`. Uses the DB-IP Lite files cached by `common/enrich.py` (downloaded on first run, ~11 MB).

In [ ]:
import polars as pl
from IPython.display import Image

from honeypot.analysis.common.enrich import enrich_ips, fetch_dbip, read_ranges
from honeypot.analysis.common.extract import get_sessions
from honeypot.analysis.geo.charts import (
    plot_bootstrap_slope,
    plot_country_ratios,
    plot_exposure_fit,
    plot_mean_variance,
    plot_residuals_vs_fitted,
    plot_rootogram,
    plot_split_methods,
    plot_split_scatter,
)
from honeypot.analysis.geo.exposure import (
    address_space,
    attacker_ips,
    country_counts,
    country_networks,
)
from honeypot.analysis.geo.regression import (
    assumption_scorecard,
    country_residuals,
    fit_exposure_model,
    frequency_table,
    mean_variance_table,
    model_checks,
    overdispersion_tests,
    pearson_residuals,
    prediction_band,
    quantile_residuals,
    slope,
    variance_by_fitted,
)
from honeypot.analysis.geo.validation import (
    DEFAULT_MIN_IPS,
    DEFAULT_REPEATS,
    bootstrap_slope,
    bootstrap_summary,
    compare_country_residuals,
    compare_slopes,
    compare_split_methods,
    cross_validate,
    drop_influential,
    fit_parts,
    influence_table,
    response_robustness,
    split_log_ratios,
)

pl.Config.set_tbl_rows(30)
pl.Config.set_fmt_str_lengths(80)

## Load the data

One row per country in the DB-IP database, **including countries that sent nothing** (dropping them would bias the slope: only the "lucky" small countries would remain). Three counts per country:
- `ips`: distinct attacking IPs (any event at all)
- `slash24s`: distinct /24 networks, so an ISP that rotates one device through many addresses in a subnet counts less
- `asns`: distinct networks (ASNs)

`addresses` is IPv4 address space from DB-IP: allocation, not use. Legacy blocks (much of the US) count in full even when they're mostly dark.

In [ ]:
MODE = "offline"

events = get_sessions(MODE)
country_ranges = read_ranges(fetch_dbip("country"), "country")
enrichment = enrich_ips(events, country_ranges, read_ranges(fetch_dbip("asn"), "asn"))
space = address_space(country_ranges)
attackers = attacker_ips(events, enrichment)
counts = country_counts(attackers, space)

print(
    f"{attackers.height:,} attacking IPs placed in {counts.height} countries; "
    f"{(counts['ips'] == 0).sum()} countries sent none"
)
counts.sort("ips", descending=True).head(10)

The raw ranking above is what a naive map would show. How much of it is just size? The rank correlation between attacking IPs and address space:

In [ ]:
counts.select(
    pl.corr("ips", "addresses", method="spearman").alias("spearman_ips_vs_addresses")
)

---
# Part 1: Which count model?

Counts of attackers are non-negative integers, so the natural starting point is a **Poisson regression**. Poisson makes one strong assumption: **variance = mean**. Given its size, a country's count should scatter around its expected value only by "coin-flip" noise.

Attack data usually breaks that. Attackers arrive in clumps: one campaign or one infected ISP adds dozens of IPs at once. When the variance is larger than the mean (**overdispersion**), Poisson still gets the trend roughly right, but its standard errors are far too small, so almost every country looks "significant".

The **negative binomial** (NB2) adds one parameter, $\alpha$: variance $= \mu + \alpha\mu^2$. With $\alpha = 0$ it is exactly Poisson, so the two models are nested and can be tested against each other.

The checks below go from model-free to formal:

| Check | What Poisson predicts | What NB predicts |
|---|---|---|
| 1. Mean vs variance in bins (no model) | variance / mean ≈ 1 everywhere | variance / mean grows with the mean |
| 2. Fit statistics | Pearson dispersion ≈ 1 | lower AIC; dispersion ≈ 1 |
| 3. Likelihood-ratio test of $\alpha = 0$ | p large | p small |
| 4. Count bands (rootogram) | expected countries per band match observed | same |
| 5. Quantile residuals vs fitted | N(0, 1) at every fitted value | N(0, 1) at every fitted value |

## 1. Mean vs variance, before any model

Countries are split into 8 equal-size bins by address space. Within a bin the countries are roughly the same size, so if Poisson held, the variance of their counts should be close to their mean. `var_to_mean` is the number to watch.

In [ ]:
mean_variance = mean_variance_table(counts)
mean_variance

On a log-log chart, Poisson is the orange line (variance = mean). The blue curve is NB2 at the $\alpha$ fitted below. Points that follow the blue curve and leave the orange line behind as the mean grows are the signature of overdispersion. Bins of tiny countries sit near both lines, because when the mean is close to 0 the $\alpha\mu^2$ term vanishes.

In [ ]:
fit = fit_exposure_model(counts)
Image(plot_mean_variance(mean_variance, fit.alpha))

## 2. Fit both models and compare

- **log-likelihood / AIC**: higher log-likelihood and lower AIC are better. AIC penalises NB for its extra parameter.
- **Pearson dispersion**: the sum of squared Pearson residuals divided by the degrees of freedom. About 1 when the variance model is right; well above 1 means the model understates the variance.
- **share of zero countries**: Poisson usually predicts far too few zeros for clumped data.

In [ ]:
model_checks(fit)

## 3. Formal tests of $\alpha = 0$

- **Likelihood ratio**: twice the gap in log-likelihood between NB and Poisson. $\alpha$ can't be negative, so the test sits on the boundary of the parameter space and the $\chi^2(1)$ p-value is halved.
- **Cameron-Trivedi**: a regression of $((y-\mu)^2 - y)/\mu$ on $\mu$ at the Poisson fit, whose slope estimates $\alpha$. It's a least-squares fit on squared residuals, so the few countries with the largest counts dominate it, and it has little power with ~250 countries. Treat a non-significant result as "inconclusive", not as support for Poisson.

In [ ]:
overdispersion_tests(fit)

## 4. Observed vs expected countries per count band

Each model implies a probability distribution for every country's count. Summing those probabilities gives the number of countries each model expects in a band: 0 attackers, 1-4, and so on. A **rootogram** draws this on a square-root axis, so small bands stay visible next to big ones. A marker far from the top of its bar is a misfit.

In [ ]:
frequencies = frequency_table(fit)
frequencies

In [ ]:
Image(plot_rootogram(frequencies))

## 5. Is the variance right at every size of country?

**The goal.** Steps 1-4 check the variance *on average*. This step checks it *across the range*: a model can get the overall dispersion right while understating the variance for big countries and overstating it for small ones. The tool is a residuals-vs-fitted plot: residuals scaled by the model's own standard deviation should spread evenly whatever the fitted count, and a band that widens to the right means the variance is understated for large counts.

**Why not Pearson residuals here.** The obvious choice is the Pearson residual $(y - \mu)/\sqrt{\text{Var}}$, and it's the right tool for the dispersion statistic in step 2. As a *picture* it breaks down for small counts, and a third of these countries sent nothing:
- At $y = 0$ the residual is fully determined by $\mu$. The zero countries form a curve on the chart rather than a scatter, so their spread says nothing about fit.
- For small $\mu$ the distribution is strongly skewed, so the familiar ±2 band no longer marks the usual ~5% tails.

**Randomized quantile residuals** (Dunn & Smyth, 1996) fix both. For each country, take where its count falls in the model's own predictive distribution (a quantile between $F(y-1)$ and $F(y)$, drawn at random within that step) and map it to a standard normal value. If the model is right, these residuals are **exactly N(0, 1) at every fitted value, zeros included**, so the plot can be read at face value:
- an even band around 0, with about 5% of points outside ±2;
- points on the diagonal of the normal QQ plot (right panel).

The random draw only spreads ties between countries with the same count; the seed is fixed, so the picture is reproducible.

In [ ]:
quantile = quantile_residuals(fit)
Image(plot_residuals_vs_fitted(quantile))

**The same check in numbers.** Group countries by fitted value and average the squared Pearson residuals in each group. Here Pearson residuals are the right tool again: under the right variance model the mean is about 1 in every group. Poisson's climbing with the fitted count is the funnel from the plot as a number. One extreme country can lift a group's mean on its own, so the median is shown next to it, only for that purpose: it sits well below 1 even under a perfect model (for a normal variable, the median of its square is 0.45, and skewed counts go lower). A mean far above a low median means one or two countries, not the whole group.

In [ ]:
variance_by_fitted(fit)

**Who is driving what's left?** If NB's Pearson dispersion in step 2 is still above 1 while the plot shows an even band, the excess comes from a few individual countries rather than a wrong variance curve. They show up again under influence in Part 3.

In [ ]:
pearson_residuals(fit).sort("resid_negbin", descending=True).head(8)

## Scorecard

Every check in one table, with the model each one favours.

In [ ]:
assumption_scorecard(fit, mean_variance)

**Reading it:** the model-free check, the likelihood-ratio test, AIC, the zero count and the dispersion should all point the same way before you rely on NB. Cameron-Trivedi is the weakest test here and shouldn't decide on its own. If they agree on NB, everything below uses the NB fit; the Poisson fit is kept only for comparison.

---
# Part 2: The fit

## The slope

`beta` is the NB estimate with its 95% Wald CI. `p_beta_eq_1` tests proportionality (attackers scale 1:1 with address space). `alpha` is the NB dispersion.

In [ ]:
pl.DataFrame([slope(fit)])

Each dot is a country; the blue line is the NB fit and the shaded band is the 95% predictive interval, i.e. where a country's count should fall if it only reflected its size. Countries outside the band are candidates for "really different". Colour marks those that stay significant after Benjamini-Hochberg correction across all countries.

In [ ]:
residuals = country_residuals(fit)
Image(plot_exposure_fit(fit, residuals, prediction_band(fit)))

## Countries furthest from the fit

`ratio` = observed / expected. `p_value` is a two-sided test of the observed count against the NB predictive distribution, `q_bh` the BH-adjusted value across all countries. Under NB the predictive distribution is wide, so few countries reach significance individually: a ratio of 5 can still be ordinary for a mid-sized country. Read the ratio and the q-value together.

In [ ]:
residuals.filter(pl.col("observed") >= DEFAULT_MIN_IPS).head(12)

In [ ]:
residuals.filter(pl.col("observed") >= DEFAULT_MIN_IPS).tail(8)

In [ ]:
Image(plot_country_ratios(residuals))

---
# Part 3: Validation

This is an **explanatory** model fitted once to ~250 countries, not a predictive one, so "does it generalize" doesn't mean a train/test accuracy score. It means: **do the conclusions survive?** Two conclusions are on the table: the slope $\beta$ (and whether it's below 1), and which countries stand out.

| Check | Question |
|---|---|
| Split stability | Do the same countries stand out in independent parts of the data? |
| Response robustness | Does it change if we count /24 networks or ASNs instead of IPs? |
| Influence | Is $\beta$ carried by a few countries? |
| Bootstrap | Is $\beta$'s CI right without trusting the model's own standard errors? |
| Cross-validation | Is a more flexible model actually better out of sample? |

## Splitting the data: which method?

There's no single right split. Each one holds a different thing apart, and the wrong one gives false comfort. The method for each split: divide the attacking IPs into parts, **refit the model on each part**, and correlate the countries' log ratios (Spearman ρ) between parts. ρ near 1 means the same countries stand out in every part.

| Method | Parts | What it tests | Pitfall |
|---|---|---|---|
| `random_ip` | each IP a coin flip | sampling noise only | **Leaks**: one campaign's IPs land on both sides, so agreement is inflated. The naive baseline. |
| `random_asn` | each ASN a coin flip | do countries stand out across *different networks*? (grouped split, like `GroupKFold`) | Strict: a country carried by one ISP must fail it. |
| `time_blocks` | IPs by first sighting, equal-size consecutive blocks | would next month look the same? | Splits are disjoint in IPs but the same campaigns can persist across time. |
| `node_role` | IPs seen by each honeypot | a second vantage point | The nodes differ in config (AWS ssh only, Azure ssh + telnet). |
| `protocol` | ssh-only vs telnet-only IPs | two attacker populations side by side | Not a replication; expect differences. |

Best practice for clustered data like this is a **grouped split**: keep each group (here, a network) on one side, so the model never "sees" the same campaign in both parts. Comparing it against `random_ip` shows how much the naive split flatters the result. For this question the time split is the most natural out-of-sample test, and the grouped split is the strictest.

In [ ]:
REPEATS = DEFAULT_REPEATS  # seeds per random split

split_summary, split_pairs = compare_split_methods(attackers, space, repeats=REPEATS)
split_summary

In [ ]:
Image(plot_split_methods(split_summary))

Per pair, with both slopes and the overlap (Jaccard) of the countries each part flags as significantly over-represented. `converged` is False when a part was too sparse for the NB fit; treat those rows with care.

In [ ]:
split_pairs.filter(pl.col("seed") == 0)

Time split, country by country. Points near the dotted diagonal stand out equally in both halves; points near the origin are ordinary in both.

In [ ]:
Image(
    plot_split_scatter(
        split_log_ratios(attackers, space, "time_blocks"), "First vs second half"
    )
)

The grouped split, country by country. If the extreme countries stay on the diagonal while the middle scatters, the extremes are genuine country-level patterns (they show up across different networks), while the middle of the ranking is driven by individual networks.

In [ ]:
Image(
    plot_split_scatter(
        split_log_ratios(attackers, space, "random_asn"), "Random ASN halves"
    )
)

Finer time resolution: four consecutive blocks instead of two. More pairs, each with fewer IPs, so ρ drops partly from noise alone.

In [ ]:
compare_split_methods(attackers, space, methods=["time_blocks"], n_blocks=4)[0]

## A different response: /24s and ASNs

Home ISPs reassign addresses, so one infected device can appear as many IPs. If a country stands out because of IP churn, it should shrink when counting /24 networks or ASNs instead. `spearman_vs_ips` compares each response's country ranking with the IP-based one (countries with ≥ 10 IPs).

In [ ]:
response_robustness(counts)

## Influence: is β carried by a few countries?

Cook's distance measures how much the fit moves when one country is left out. Then β is refitted without the k most influential countries. If the CI for β crosses 1 after dropping a handful, "β < 1" is fragile.

In [ ]:
influence_table(fit).head(10)

In [ ]:
drop_influential(counts, fit)

## Bootstrap over countries

**Why.** Part 2's CI for β comes from the NB model itself: it's only right if the model's assumptions are right, and we already know three countries don't fit well. The bootstrap gets a CI that doesn't lean on those assumptions. The ~250 countries are treated as a sample: draw 250 of them *with replacement* (some twice, some not at all), refit, record β, and repeat 500 times. The spread of those 500 β values shows how much the estimate depends on which countries happen to be in the data.

**How to read it.** If the bootstrap CI matches the Wald CI from Part 2, the model's own uncertainty is trustworthy. If it's wider, trust the bootstrap: a few influential countries are carrying more of the estimate than the model admits. Check especially whether the bootstrap CI still excludes 1.

In [ ]:
N_BOOT = 500

draws = bootstrap_slope(counts, n_boot=N_BOOT)
bootstrap_summary(draws)

In [ ]:
Image(plot_bootstrap_slope(draws, slope(fit)["beta"]))

## Cross-validation: choosing between model specifications

**Why.** A more flexible model always fits the data it was fitted on better, so in-sample fit can't decide whether an extra term (here, a quadratic on log size) is real or just bends the line towards noise. Cross-validation asks the fairer question: *does the model predict countries it hasn't seen?* Repeated 5-fold over countries: fit on 80% of the countries, score the held-out 20% by log-likelihood (how probable the model found their actual counts), and repeat 10 times with new folds. It's for choosing between specifications (Poisson, NB, NB + quadratic), not a performance claim: no one will use this model to predict a new country.

**How to read it.** Compare models on `diff_vs_negbin` (the paired, per-fold difference) and `share_folds_better`, not on the raw means: the fold-to-fold spread (`sd`) mostly reflects which countries landed in a fold, and it's shared by every model. A quadratic term that wins most folds means the log-log line bends: the slope differs between small and large address holders.

In [ ]:
cross_validate(counts)

In [ ]:
fit_exposure_model(counts, quadratic=True).negbin.summary().tables[1]

---
# Part 4: Follow-ups

Three questions the results above raise: is a country's excess real or an artefact of how IPs are counted, which networks carry it, and do SSH and Telnet attackers (mostly IoT botnets, see `../../network/README.md`) behave differently?

## Does the ranking survive counting /24 networks?

A country can look over-represented because one device or one small network shows up under many IP addresses (an ISP reassigning addresses, a hosting provider with many IPs in one subnet). Counting distinct /24 networks instead of IPs collapses that. A ratio that stays high under `slash24s` is spread over many networks; one that drops a lot was concentrated in a few.

In [ ]:
response_fits = {r: fit_exposure_model(counts, r) for r in ["ips", "slash24s"]}
compare_country_residuals(response_fits).head(12)

## Which networks carry the over-represented countries?

`top_share` is the share of a country's attacking IPs in its single largest network, `ips_per_slash24` how densely they sit in subnets. A country with `top_share` near 1 is really one network, often a hosting provider, so the finding is about that network, not the country. Many ASNs with a low `top_share` means a country-wide pattern.

In [ ]:
networks = country_networks(attackers)
(
    residuals.filter(pl.col("observed") >= DEFAULT_MIN_IPS)
    .head(10)
    .select("country", "ratio", "q_bh")
    .join(networks, on="country")
    .sort("ratio", descending=True)
)

## SSH vs Telnet: different populations, different slopes?

The protocol split refits the model separately for IPs that only used SSH and IPs that only used Telnet (IPs that used both are left out, so the two populations are disjoint and their slopes independent). `compare_slopes` gives each slope with a test of β = 1, then a z-test of equal slopes.

In [ ]:
protocol_fits = fit_parts(attackers, space, "protocol")
compare_slopes(protocol_fits)

The countries significant (q < 0.05) under either protocol, side by side: does a country stand out for one protocol and not the other?

In [ ]:
compare_country_residuals(protocol_fits).filter(
    (pl.col("q_ssh") < 0.05) | (pl.col("q_telnet") < 0.05)
)

## Exploration

Ad hoc from here. Still open:
- Covariates beyond address space (internet users, GDP per capita from the World Bank). Judge them with `cross_validate`: an extra variable should improve held-out log-likelihood, not just the in-sample fit.
- A residual choropleth (needs country shapes, e.g. Natural Earth).